In [1]:
# 1D let the domain be uniform and defined on [0, 1]
import numpy as np
from scipy.linalg import eigh
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-poster")

In [2]:

# method for putting values on the main diagonal
def off_diag_values_matrix(rank, value):
       # the size should be smaller because there are less numbers off diagonal than on main diagonal
       diag = np.full(rank - 1, value)
       lower_off_centre_diag = np.diag(diag, k=-1)
       higher_off_centre_diag = np.diag(diag, k=1)

       matrix = lower_off_centre_diag + higher_off_centre_diag
       
       return matrix

# stiffness matrix
def stiffness_matrix_A(rank, h):
       # create a matrix (interior_nodes x interior_nodes) of zeros
       A = np.zeros((rank, rank), dtype=float)
       # diagonal values
       a_ii = 2 / h
       np.fill_diagonal(A, a_ii)
       # off diagonal values
       a_ij = -1 / h

       A = A + off_diag_values_matrix(rank, a_ij)
       return A

# mass matrix
def mass_matrix_M(rank, h):
       # create a matrix (interior_nodes x interior_nodes) of zeros
       M = np.zeros((rank, rank), dtype=float)
       # diagonal values
       m_ii = (4 * h) / 6
       np.fill_diagonal(M, m_ii)
       # off diagonal values
       m_ij = h / 6

       M = M + off_diag_values_matrix(rank, m_ij)
       return M

# error of all eigenvalues
# takes 2 lists of real eigenvalues and computed
def all_eigval_error(real_eigvals, comp_eigvals):
       # for values in both arrays
       for real_val, val in zip(real_eigvals, comp_eigvals):
              error = abs(real_val - val)
              print(
                     f"Computed eigenvalue: {val:.5f}\n",
                     f"Real value: {real_val:.5f}\n",
                     f"Error: {error:.5f}"
              )

# error of the first eigenvalue
# will get only the first elements of the arrays
def first_eigval_error(real_eigval, comp_eigval):
       error = abs(real_eigval - comp_eigval)
       return error

def plotting_eigenfunction(x, u):
       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(x, u, marker="o")
       ax.set_xlabel("x")
       ax.set_ylabel(r"$\phi_1^h(x)$")
       ax.set_title("First Eigenfunction")
       ax.grid(True)

def comparing_exact_with_comp_functions(x, u_comp, x_of_exact):
       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(x, u_comp, "b", label="Computed function")
       ax.plot(x_of_exact, np.sin(np.pi*x_of_exact), "ro", label="Exact function")
       ax.set_xlabel("x")
       ax.set_ylabel("u(x)")
       ax.legend()
       ax.grid(True)
       



In [ ]:
# Solver of 1D FEM 
def solve_eigenproblem(num_elements):
       # define step
       h = 1 / num_elements
       # define interior nodes
       interior_nodes = num_elements - 1

       # Stiffness matrix 
       A = stiffness_matrix_A(interior_nodes, h)
       # Mass matrix
       M = mass_matrix_M(interior_nodes, h)

       # find eigenvalues
       eigvals, eigvecs = eigh(A, M)
       # real eigenvalue
       real_eigval = (np.pi)**2

       # Error
       error = first_eigval_error(real_eigval, eigvals[0])
       
       df = pd.DataFrame([[eigvals[0], real_eigval, error]], columns=["Computed eigenvalue", "Real eigenvalue", "Error"], index=[f"For {num_elements} elements"])
       print(df)

       # plotting the eigenfunction
       u = eigvecs[:, 0]
       x = np.linspace(h, 1-h, interior_nodes)
       # preventing the upside down shape
       if u[len(u) // 2] < 0:
              u = -u
       plotting_eigenfunction(x, u)

       # plotting the eigenfucntion with boundary values
       u_full = np.concetenate([0], u, [0])
       x_full = np.linspace(0, 1, num_elements + 1)
       # for 16 elements we have 17 nodes and 15 interior nodes


       x_of_exact = np.linspace(0, 1, 500)
       comparing_exact_with_comp_functions(x, u, x_of_exact)
       plt.show()

solve_eigenproblem(16)     
       